# NB10：主成分分析 PCA — 六種食用油分類（Principal Component Analysis）

**食品分析實驗 週次 12｜對應課堂教材：`lab_decks/w12_pca`**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 直覺理解 PCA 在找什麼：資料**變異最大**的方向，PC1 ⟂ PC2 ⟂ …。
2. 分清楚 **scores（分數）**＝樣品在圖上的位置，與 **loadings（負荷量）**＝哪些波數在驅動這個 PC。
3. 用 **scikit-learn 的 `PCA`** 與**用 numpy 手刻 SVD** 各做一次 PCA，驗證兩者結果一致（差在正負號）。
4. 說出為什麼光譜資料通常 **mean-centering 一定做，但不做 autoscale**。
5. 看懂 **scree plot**（陡坡圖）與解釋變異 (explained variance)，並判讀 loadings 對應的化學意義（不飽和相關譜帶，約 1265 / 1655 cm⁻¹，需保留「約略」的保留態度）。
6. 用分數圖找出**離群樣品**，並用 **Hotelling T²** 做初步的離群值判斷。
7. 觀察前處理（SNV／導數）對分數圖的影響，並知道同一份分析在 R 與 Orange 中怎麼做。

## 如何使用本筆記本

- 直接「全部執行」（執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬光譜**（非真實量測，模擬拉曼光譜）。
- `load_spectra()` 用法與 NB09 一致，支援本機/雲端路徑、已發布的 Google 試算表連結，或退回內建模擬資料。
- 若要換成你們班用 SpectraView 匯出的實際光譜，可用「Export combined CSV → rows」匯出，`split_spectra` 支援這個格式（見下方「光譜資料格式」）。

> ⚠️ **本筆記本內建的資料是模擬拉曼光譜**：用「不飽和度」與「類胡蘿蔔素含量」兩個參數疊加高斯峰生成，
> 波峰位置為教學示意，不是文獻定量值。做作業時請換成你們班的實際量測光譜。

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

## 1. 環境設定 (Setup)

只使用 numpy / pandas / scipy / matplotlib / scikit-learn，不需要額外安裝套件。

**注意**：所有圖表座標軸與圖例一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 7
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded demo data (fallback) -------------------------------------
# Exact content of data/nb10_demo_spectra.csv, embedded so the notebook is
# self-contained. 六種食用油 x 3 重複的模擬拉曼光譜。
SAMPLE_CSV = """sample_id,group,instrument,rep,oil,oil_zh,900.000,915.254,930.508,945.763,961.017,976.271,991.525,1006.780,1022.034,1037.288,1052.542,1067.797,1083.051,1098.305,1113.559,1128.814,1144.068,1159.322,1174.576,1189.831,1205.085,1220.339,1235.593,1250.847,1266.102,1281.356,1296.610,1311.864,1327.119,1342.373,1357.627,1372.881,1388.136,1403.390,1418.644,1433.898,1449.153,1464.407,1479.661,1494.915,1510.169,1525.424,1540.678,1555.932,1571.186,1586.441,1601.695,1616.949,1632.203,1647.458,1662.712,1677.966,1693.220,1708.475,1723.729,1738.983,1754.237,1769.492,1784.746,1800.000
EVOO_r1,0,Raman785,1,EVOO,特級初榨橄欖油,0.00532,0.01265,-0.01143,0.00423,0.00402,-0.0311,0.02537,0.02493,0.03734,0.1585,0.38683,0.60395,0.67581,0.49592,0.29205,0.15068,0.38571,0.53206,0.20976,0.01619,0.00099,0.09417,0.21649,0.48312,0.72118,0.67564,0.61249,0.38151,0.15639,0.06599,-0.02649,0.01246,-0.01905,0.15931,0.62156,1.20527,1.07928,0.44565,0.13642,0.23832,0.77309,0.86894,0.32954,0.01579,0.01114,0.06163,0.08719,0.2441,0.5761,0.78477,0.78042,0.54123,0.26555,0.14913,0.17482,0.27125,0.19811,0.10811,0.00963,-0.03987
EVOO_r2,0,Raman785,2,EVOO,特級初榨橄欖油,-0.02466,-0.03422,-0.03971,-0.03627,-0.02069,-0.0376,0.01765,-0.00972,0.03386,0.16754,0.37338,0.56036,0.63981,0.50087,0.25149,0.09454,0.36408,0.54265,0.16767,-0.01366,-0.00253,0.03287,0.19485,0.50717,0.71861,0.69465,0.5697,0.38163,0.16116,0.04039,-0.00691,0.00163,0.06811,0.18365,0.60907,1.11048,0.97802,0.44206,0.1275,0.20759,0.74014,0.83578,0.38272,0.06214,0.04818,0.01792,0.08384,0.26028,0.54492,0.76563,0.72882,0.52804,0.27127,0.15586,0.19795,0.30133,0.20122,0.08299,0.04957,0.02238
EVOO_r3,0,Raman785,3,EVOO,特級初榨橄欖油,0.00196,-0.0003,0.012,-0.00135,-0.01908,-0.01362,-0.01541,0.00713,0.01685,0.10862,0.3514,0.54405,0.60004,0.49913,0.24614,0.14463,0.34483,0.5066,0.13819,-0.00221,-0.00111,0.03089,0.1977,0.45288,0.69051,0.64728,0.55634,0.38308,0.18382,0.07637,-0.02472,0.01746,0.04628,0.18491,0.59637,1.12724,1.0312,0.4026,0.12663,0.25591,0.71373,0.85585,0.32913,0.03859,-0.00702,0.02612,0.0827,0.2681,0.4996,0.74615,0.76686,0.52927,0.23751,0.17054,0.19385,0.26633,0.20693,0.11579,0.0303,-0.02649
ROO_r1,0,Raman785,1,ROO,精製橄欖油,-0.01248,0.0157,-0.00135,-0.02155,0.01534,-0.02211,-0.04569,0.04425,-0.00115,0.15248,0.35494,0.55164,0.58066,0.49294,0.22064,0.06826,0.06104,0.09287,0.02753,0.01563,-0.00473,0.04213,0.17272,0.4585,0.67526,0.65061,0.54796,0.34333,0.14384,0.04123,0.00226,0.01403,0.02484,0.19365,0.58661,1.07088,0.93447,0.41536,0.10642,0.0285,0.10649,0.13849,0.05064,0.0006,0.02615,0.0232,0.08905,0.26264,0.52102,0.71256,0.7079,0.50344,0.25093,0.11296,0.22085,0.24877,0.20765,0.06421,0.05245,-0.00179
ROO_r2,0,Raman785,2,ROO,精製橄欖油,0.01068,0.00894,0.00936,0.02136,0.01333,0.03127,0.00304,-0.00034,0.0539,0.17286,0.38862,0.6002,0.6161,0.51093,0.29989,0.11076,0.07054,0.09549,0.02074,0.01987,0.01101,0.08146,0.21867,0.47347,0.70572,0.70455,0.5509,0.36212,0.11683,0.06048,-0.00572,-0.04128,-0.01509,0.15809,0.62625,1.11208,0.99767,0.40376,0.06337,0.05577,0.11388,0.11999,0.07176,-0.0175,-0.01208,0.03127,0.10899,0.236,0.53782,0.72158,0.73835,0.49818,0.21534,0.14486,0.20376,0.23655,0.17971,0.09883,-0.01813,-0.0412
ROO_r3,0,Raman785,3,ROO,精製橄欖油,0.03889,0.01775,-0.00613,-0.00154,0.06757,-0.00215,-0.00449,0.02396,0.06483,0.16082,0.33837,0.54999,0.65154,0.49301,0.23244,0.13024,0.09421,0.08224,0.02438,0.01095,-0.00324,0.02523,0.21124,0.45074,0.6897,0.64545,0.55083,0.36167,0.19582,0.0204,-0.02233,-0.01168,-0.00237,0.1483,0.60968,1.11133,1.00557,0.40452,0.14201,0.07766,0.0989,0.0735,0.04447,0.00378,-0.0487,0.013,0.06806,0.21732,0.50961,0.74288,0.71127,0.49933,0.23134,0.12312,0.18516,0.24761,0.16851,0.02723,0.02022,-0.04117
SOY_r1,0,Raman785,1,SOY,大豆油,-0.01813,-0.04098,-0.04581,0.00743,-0.01571,-0.00386,0.0056,-0.0056,0.02523,0.12028,0.2835,0.53759,0.63559,0.48552,0.26375,0.08781,0.03634,0.03903,-0.01577,-0.01084,-0.02905,0.07916,0.26391,0.63328,0.89662,0.8173,0.5836,0.37285,0.12337,0.04779,0.00893,-0.02964,0.04932,0.14917,0.61806,1.09048,0.99654,0.45199,0.07866,-0.00457,0.0406,0.06623,0.01761,-0.00117,0.02233,-0.02219,0.10147,0.30246,0.68859,0.96947,0.96063,0.67531,0.31677,0.18966,0.2046,0.26203,0.19678,0.11946,0.03555,0.03107
SOY_r2,0,Raman785,2,SOY,大豆油,0.01946,0.01477,0.01379,-0.00073,-0.00156,0.02253,0.05132,-0.00141,0.06141,0.15149,0.38175,0.59912,0.68045,0.45293,0.26008,0.08435,0.04326,0.02008,-0.00072,0.01775,0.04955,0.01137,0.26636,0.64213,0.87998,0.80129,0.56708,0.34427,0.16946,0.06148,0.00771,-0.01529,0.02252,0.15711,0.6178,1.10814,0.96894,0.42749,0.06519,0.03433,0.05807,0.01606,0.02269,-0.02052,0.02134,0.03779,0.09682,0.33812,0.66504,0.96465,0.96397,0.66789,0.32351,0.17429,0.18665,0.26524,0.2104,0.0681,-0.01557,0.00923
SOY_r3,0,Raman785,3,SOY,大豆油,-0.0241,0.00691,0.03202,-0.03606,0.01098,-0.00479,-0.01453,0.02043,0.02093,0.16327,0.3951,0.60126,0.65066,0.47947,0.2483,0.06608,0.07529,0.02745,-0.01113,-0.02411,-0.01173,0.06055,0.25056,0.68036,0.88085,0.80728,0.6111,0.39865,0.18412,0.03088,0.01111,0.00478,0.03217,0.17096,0.60084,1.10334,1.00546,0.40993,0.11967,0.0212,0.01691,0.03654,0.00719,-0.00032,0.02205,-0.00534,0.06973,0.33509,0.68737,1.00417,1.00782,0.68495,0.35743,0.16193,0.20959,0.2439,0.177,0.09799,0.03097,-0.0376
CAN_r1,0,Raman785,1,CAN,芥花油,0.01848,-0.01614,-0.0064,-0.03089,0.0477,0.00719,0.01676,0.01955,0.05926,0.15609,0.33501,0.57062,0.62331,0.50472,0.23893,0.08315,0.05164,0.05402,0.0103,-0.01415,0.00452,0.04682,0.27528,0.57674,0.8059,0.73902,0.54644,0.37279,0.18351,0.05052,0.00534,0.02355,0.01329,0.17214,0.57057,1.04515,0.95671,0.41659,0.12646,0.01497,0.03704,0.01909,-0.01931,-0.03146,0.01411,-0.00094,0.11732,0.31421,0.59295,0.89936,0.86918,0.61973,0.29449,0.18307,0.17774,0.25946,0.18129,0.08159,0.03523,0.00906
CAN_r2,0,Raman785,2,CAN,芥花油,0.00879,-0.03473,0.00439,0.00688,-0.02132,0.00179,-0.03105,0.03213,0.03405,0.14407,0.37395,0.56829,0.6514,0.47103,0.24826,0.06708,0.03355,0.04817,0.01045,0.01627,-0.01477,0.03974,0.23467,0.58745,0.81233,0.75587,0.58763,0.35729,0.15467,0.06617,-0.00563,0.00586,0.05398,0.16449,0.61861,1.08784,1.01619,0.37925,0.06979,0.00154,0.07167,0.05196,0.01523,-0.01939,0.01438,0.02785,0.08639,0.26776,0.59875,0.90145,0.90194,0.62123,0.30895,0.13724,0.20752,0.23807,0.20134,0.10021,0.01394,-0.01041
CAN_r3,0,Raman785,3,CAN,芥花油,-0.01625,-0.06341,-0.00426,0.00942,0.0194,-0.01636,-0.01439,-0.00236,0.05972,0.1296,0.36163,0.57347,0.66769,0.47892,0.27761,0.09001,0.03628,0.04064,0.0306,-0.03398,0.02729,0.07156,0.24694,0.62345,0.83558,0.80693,0.58226,0.3549,0.18478,0.08471,0.0189,0.0199,0.01262,0.14069,0.61876,1.16456,0.97035,0.42694,0.056,0.02422,0.0566,0.01994,0.01125,0.03113,0.00359,-0.0005,0.07491,0.26448,0.6537,0.90843,0.94272,0.65409,0.30575,0.16442,0.22543,0.26656,0.20871,0.05717,0.04243,-0.01113
COC_r1,0,Raman785,1,COC,椰子油,0.01461,0.0479,-0.00171,-0.02105,0.04781,0.00844,0.00085,-0.0085,0.02562,0.17461,0.33006,0.53082,0.59471,0.49812,0.23751,0.10642,0.03711,0.02121,-0.01157,0.01564,0.03469,0.00475,0.00152,0.03394,0.15935,0.32261,0.45915,0.38417,0.15208,0.06384,0.01177,0.00016,0.02575,0.18327,0.61802,1.10692,1.02092,0.4223,0.07548,-0.01968,0.01197,-0.00763,-0.00651,0.00714,-0.0021,0.0185,0.00889,0.00728,0.02951,0.04366,0.06653,0.03755,0.02101,0.0854,0.16552,0.21825,0.14507,0.08155,0.01789,-0.00255
COC_r2,0,Raman785,2,COC,椰子油,0.0038,0.01268,0.00521,0.00459,-0.00215,0.01382,0.01254,0.00011,0.06491,0.17069,0.37276,0.5721,0.63784,0.47216,0.28007,0.05757,0.02667,0.01136,0.02415,0.00324,-0.01604,-0.02188,0.00697,0.06583,0.16617,0.34375,0.47502,0.40257,0.21237,0.02908,0.00814,0.03403,0.0333,0.16515,0.66635,1.17496,1.05628,0.48705,0.13267,-0.02251,0.02587,0.05238,-0.00757,0.0024,0.02633,-0.01274,0.01099,0.01402,-0.0014,0.06272,0.05184,0.05657,0.07145,0.08479,0.16442,0.24845,0.18809,0.09588,-0.01663,-0.02892
COC_r3,0,Raman785,3,COC,椰子油,-0.00645,-0.01867,-0.02016,-0.0169,-0.0238,-0.01136,-0.01356,0.03154,0.04354,0.12118,0.3181,0.56177,0.61104,0.46916,0.25906,0.12022,0.00205,-0.00357,-0.03359,0.00801,-0.00899,-0.01114,0.00169,0.07132,0.15153,0.31793,0.48822,0.36475,0.15699,0.10025,-1e-05,-0.00858,0.00776,0.16466,0.63894,1.1494,1.01838,0.43163,0.06438,0.02126,0.04664,0.03959,0.00574,0.01981,-0.00445,0.00351,-0.01501,0.01645,0.04688,0.05893,0.07793,0.08402,0.06525,0.07268,0.16036,0.22677,0.16806,0.08546,0.0335,0.01234
PEA_r1,0,Raman785,1,PEA,花生油,-0.00422,-0.01638,0.03302,0.01953,-0.01952,-0.0193,-0.01174,0.00842,0.06849,0.17111,0.37799,0.60111,0.67012,0.5139,0.27386,0.09615,0.00578,0.02763,0.0443,-0.01033,-0.00679,0.06575,0.19117,0.44056,0.6799,0.67151,0.57441,0.3651,0.17646,0.05091,0.02203,0.02498,0.04447,0.17048,0.63714,1.11197,1.02061,0.4503,0.08599,0.0057,0.03638,0.01876,0.02501,0.00947,-0.01545,0.01852,0.0963,0.24802,0.49573,0.6852,0.68022,0.46698,0.25288,0.1939,0.22645,0.28676,0.22386,0.05402,0.0181,-0.00428
PEA_r2,0,Raman785,2,PEA,花生油,-0.03016,-0.01554,-0.0046,-0.02494,-0.02716,-0.02745,0.00553,-0.02181,0.05627,0.1467,0.37281,0.61894,0.64597,0.49002,0.27363,0.0735,0.02036,0.02243,0.00571,-0.02981,-0.00665,0.05409,0.21851,0.42179,0.69241,0.69506,0.55341,0.35762,0.17932,0.03612,0.00966,0.02,0.01652,0.17737,0.65478,1.19117,1.07794,0.48814,0.10591,0.02204,0.09143,0.06308,0.03374,-0.01125,0.02663,0.01068,0.07673,0.21639,0.48975,0.72121,0.68937,0.48306,0.2587,0.13851,0.19235,0.25971,0.18877,0.11544,0.01778,0.03508
PEA_r3,0,Raman785,3,PEA,花生油,0.08436,0.07332,-0.02592,0.07712,0.09542,-0.10389,0.01734,-0.04651,-0.03909,0.22329,0.35873,0.61722,0.94189,0.93964,0.93466,1.10447,0.87095,0.93841,0.66166,0.60224,0.4272,0.3771,0.1926,0.45293,0.78173,0.63346,0.64495,0.26899,0.29698,0.14622,0.04612,-0.10811,0.00168,0.20366,0.68012,1.14918,1.01715,0.47974,0.25868,-0.14959,-0.11004,-0.12124,-0.02639,-0.14535,0.08307,-0.02646,0.03923,0.09039,0.50763,0.71313,0.72655,0.39957,0.32693,-0.01346,0.33702,-0.04773,0.13675,-0.03545,0.0907,-0.17946
"""

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
df_raw = load_spectra()
meta, X, axis = split_spectra(df_raw)
print(f"讀入 {len(meta)} 條光譜，x 軸範圍 {axis.min():.0f}-{axis.max():.0f} cm-1，"
      f"共 {len(axis)} 個資料點，{meta['oil'].nunique()} 種油。")
meta.head(6)

In [ ]:
plot_spectra(axis, X, meta["oil"].tolist(),
             title="Six edible oils: raw simulated Raman-like spectra",
             xlabel="Raman shift (cm-1)")
plt.show()

## 2. 前處理：mean-centering 一定做，autoscale 通常不做

- **Mean-centering**：每個波數都減去自己的平均值，PCA 才是在找「相對平均光譜的變異方向」，
  幾乎是所有 PCA 實作的必要前處理（`sklearn.decomposition.PCA` 內部也會自動做）。
- **Autoscale**（每個波數再除以自己的標準差）：光譜資料通常不建議做，因為那些幾乎只有雜訊的波數，
  標準差本身很小，除下去反而被「放大」成跟真正有訊號的波數差不多的量級，稀釋掉有用的變異方向。

下面示範：對同一組資料分別做「只 mean-center」與「mean-center + autoscale」，比較兩者在
一段**幾乎只有雜訊**的波數區間（例如 1750–1800 cm⁻¹，訊號很弱）上的相對權重。

In [ ]:
Xc = X - X.mean(axis=0)
Xc_autoscale = Xc / (Xc.std(axis=0, ddof=0) + 1e-9)

noise_region = axis >= 1750
print(f"僅 mean-center：雜訊區段(>=1750cm-1)佔全部欄位標準差總和的比例 = "
      f"{Xc.std(axis=0)[noise_region].sum() / Xc.std(axis=0).sum():.4f}")
print(f"mean-center+autoscale：同一個比例 = "
      f"{Xc_autoscale.std(axis=0)[noise_region].sum() / Xc_autoscale.std(axis=0).sum():.4f}")
print("-> autoscale 後，雜訊區段對整體變異的『占比』被明顯放大，這正是光譜 PCA 通常不做 autoscale 的原因。")

## 3. PCA：scikit-learn 版 vs. numpy SVD 手刻版

`sklearn.decomposition.PCA` 內部本質上也是對 mean-centered 資料做 SVD。這裡刻意兩種都做一次，
驗證兩者的 scores 與 loadings 一致——**差別只在正負號**（PCA 的方向本身沒有固定正負，
只要兩個實作對同一個 PC 選的方向剛好相反，畫出來的分數圖會左右或上下鏡射，但形狀、群聚關係完全一樣）。

In [ ]:
# ---- (a) scikit-learn ----
pca = PCA(n_components=5)
sk_scores = pca.fit_transform(Xc)          # already mean-centers internally
sk_loadings = pca.components_               # shape (n_components, n_features)
sk_explained = pca.explained_variance_ratio_ * 100

# ---- (b) numpy SVD ----
U, Sv, Vt = np.linalg.svd(Xc, full_matrices=False)
n_samp = Xc.shape[0]
eigvals = Sv ** 2 / (n_samp - 1)
np_explained = eigvals / eigvals.sum() * 100
np_scores = U * Sv
np_loadings = Vt

# ---- align signs component-by-component before comparing ----
for k in range(2):
    sign = np.sign(np.dot(sk_scores[:, k], np_scores[:, k]) + 1e-12)
    agree = np.allclose(sk_scores[:, k] * sign, np_scores[:, k], atol=1e-6)
    print(f"PC{k+1}: explained_variance sklearn={sk_explained[k]:.2f}%  numpy={np_explained[k]:.2f}%  "
          f"scores agree (up to sign): {agree}")

兩者的 explained variance 應該幾乎一樣（微小差異來自浮點數計算路徑不同），
scores 對齊正負號後也應該幾乎完全相同。之後的分析我們統一用 numpy SVD 版本的結果
（`np_scores` / `np_loadings` / `np_explained`），並依照下面的規則把正負號固定住，
方便所有人畫出來的圖方向一致：

In [ ]:
# 固定正負號慣例：PC1 在 1655cm-1(不飽和)為正、PC2 在 1520cm-1(類胡蘿蔔素)為正
idx1655 = int(np.argmin(np.abs(axis - 1655)))
if np_loadings[0, idx1655] < 0:
    np_loadings[0, :] *= -1
    np_scores[:, 0] *= -1
idx1520 = int(np.argmin(np.abs(axis - 1520)))
if np_loadings[1, idx1520] < 0:
    np_loadings[1, :] *= -1
    np_scores[:, 1] *= -1

PC1, PC2 = np_scores[:, 0], np_scores[:, 1]
oils = meta["oil"].to_numpy()
print("PC1/PC2 sign convention fixed.")

## 4. 分數圖 (Score Plot)：PC1 vs PC2

每個點代表一條光譜；同一種油的三次重複應該緊緊聚在一起。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.get_cmap("tab10")
for i, oil in enumerate(OILS_ORDER := sorted(set(oils), key=list(oils).index)):
    m = oils == oil
    ax.scatter(PC1[m], PC2[m], s=70, color=colors(i), label=oil, edgecolor="black", linewidth=0.5)
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel(f"PC1 ({np_explained[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({np_explained[1]:.1f}%)")
ax.set_title("Score plot: six edible oils (3 reps each)")
ax.legend(title="oil", fontsize=9)
plt.show()

**判讀重點**：
- **PC1**（不飽和相關）：椰子油 (COC) 的飽和度最高，應該落在 PC1 最負的一端。
- **PC2**（類胡蘿蔔素相關）：特級初榨橄欖油 (EVOO) 因為類胡蘿蔔素譜帶，通常跟其他油分得開。
- **重複性 (QC)**：多數油的三個重複應該緊聚；但花生油 (PEA) 有一個重複明顯偏離同組其他兩點——
  這就是「離群分數」，第 7 節會回頭檢查。
- PCA 是**非監督式 (unsupervised)** 方法：它完全不知道每個點是哪一種油，只根據光譜本身的變異來分——
  所以如果分數圖上不同油真的分開，代表光譜本身存在真實的化學差異。

## 5. Loadings：哪些波數驅動了 PC1 / PC2

**Loading** 描述「每個波數對某個主成分的貢獻」。PC1 loading 在約 **1655 cm⁻¹**(C=C 伸縮)與
約 **1265 cm⁻¹**(=C–H 彎曲，順式不飽和)附近應該是正值高峰——這兩個都是**不飽和**相關的拉曼譜帶，
可以解釋椰子油(高度飽和)為什麼在 PC1 上跟其他油差最遠。這些波段歸屬請視為**約略**指認，
實際樣品仍須以標準品或文獻圖譜比對確認。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
axes[0].plot(axis, np_loadings[0], color="tab:blue")
axes[0].axvline(1655, color="grey", linestyle=":", label="~1655 (C=C stretch)")
axes[0].axvline(1265, color="grey", linestyle="--", label="~1265 (=C-H bend)")
axes[0].set_ylabel(f"PC1 loading ({np_explained[0]:.1f}%)")
axes[0].legend(fontsize=8)

axes[1].plot(axis, np_loadings[1], color="tab:orange")
axes[1].axvline(1520, color="grey", linestyle=":", label="~1520 (carotenoid C=C)")
axes[1].set_ylabel(f"PC2 loading ({np_explained[1]:.1f}%)")
axes[1].set_xlabel("Raman shift (cm-1)")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Scree Plot：解釋變異隨 PC 遞減

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
n_show = 5
ax.bar(range(1, n_show + 1), np_explained[:n_show], color="tab:blue")
ax.set_xticks(range(1, n_show + 1))
ax.set_xticklabels([f"PC{k}" for k in range(1, n_show + 1)])
ax.set_ylabel("Explained variance (%)")
ax.set_title("Scree plot")
plt.show()

print(f"PC1+PC2 累積解釋變異 = {np_explained[0] + np_explained[1]:.1f}%")

## 7. 離群樣品：從分數圖回頭檢查原始光譜 + Hotelling T² 預覽

分數圖上有一個花生油的重複明顯偏離同組其他兩點。標準流程：**先回頭看原始光譜**，
確認是不是氣泡、比色管沒對準、雜散光等量測問題，而不是急著刪除或直接接受。

**Hotelling T²** 是一個更系統化的「多變量離群值」指標：用前 k 個 PC 的分數，計算
$T^2 = \sum_{a=1}^{k} t_a^2/\lambda_a$（$\lambda_a$ 為第 a 個 PC 的特徵值），
再用 F 分布算出一個 95% 信心界線。這裡只做一個**預覽**——嚴謹的 T²/Q 殘差分析、
以及要不要把離群樣品排除後重新建模，會在之後的章節更完整地討論。

In [ ]:
outlier_mask = (oils == "PEA") & (meta["rep"].to_numpy() == 3)
print("離群候選：", meta.loc[outlier_mask, "sample_id"].tolist())

fig, ax = plt.subplots(figsize=(9, 5))
pea_mask = oils == "PEA"
for i in np.where(pea_mask)[0]:
    style = dict(color="red", linewidth=2.2) if outlier_mask[i] else dict(color="tab:blue", linewidth=1.2, alpha=0.7)
    ax.plot(axis, X[i], label=meta["sample_id"].iloc[i], **style)
ax.set_xlabel("Raman shift (cm-1)"); ax.set_ylabel("Intensity (a.u.)")
ax.set_title("PEA replicates: outlier (red) vs normal (blue)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
def hotelling_t2(scores_k, eigvals_k):
    return np.sum((scores_k ** 2) / eigvals_k, axis=1)


k = 2
lambdas_k = eigvals[:k]
scores_k = np_scores[:, :k]
T2 = hotelling_t2(scores_k, lambdas_k)

n = scores_k.shape[0]
alpha = 0.05
T2_limit = k * (n - 1) / (n - k) * stats.f.ppf(1 - alpha, k, n - k)

for i in np.argsort(-T2)[:5]:
    flag = "  <-- 離群候選" if T2[i] > T2_limit else ""
    print(f"{meta['sample_id'].iloc[i]:>10s}: T2 = {T2[i]:7.2f}{flag}")
print(f"\nT2 95% 界線 (k={k} PCs) = {T2_limit:.2f}")

注意：這個離群樣品目前**還在模型裡**，代表它本身也影響了 PCA 的計算結果（尤其是 PC2）。
更嚴謹的做法通常是：**先回頭檢查原始光譜與量測記錄**（氣泡、比色管沒對準、雜散光、記錄本上是否有異常備註），
**確認**這確實是量測異常而不是真實的化學差異、並**記錄下判定的理由**，才決定要把這個樣品**移除或重新量測**，
然後重新配適模型——不是看到 T² 超標就直接刪掉。這裡先用「預覽」版本建立直覺即可，實際的移除／重新量測與
重新配適會在下一節與練習題示範。

## 8. 前處理（SNV／導數）對分數圖的影響

目前的分數圖是對**原始強度**做 PCA。如果先做 SNV 或一階導數，會發生什麼事？

In [ ]:
def snv(y):
    y = np.asarray(y, dtype=float)
    return (y - y.mean()) / y.std(ddof=0)


X_snv = np.array([snv(y) for y in X])
X_deriv1 = savgol_filter(X, window_length=9, polyorder=2, deriv=1, delta=(axis[1] - axis[0]), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for name, Xp, ax in [("SNV", X_snv, axes[0]), ("1st derivative", X_deriv1, axes[1])]:
    Xc_p = Xp - Xp.mean(axis=0)
    Up, Svp, Vtp = np.linalg.svd(Xc_p, full_matrices=False)
    scores_p = Up * Svp
    explained_p = (Svp ** 2 / (Xc_p.shape[0] - 1))
    explained_p = explained_p / explained_p.sum() * 100
    for i, oil in enumerate(OILS_ORDER):
        m = oils == oil
        ax.scatter(scores_p[m, 0], scores_p[m, 1], s=60, color=colors(i), label=oil)
    ax.set_xlabel(f"PC1 ({explained_p[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({explained_p[1]:.1f}%)")
    ax.set_title(f"Score plot after {name}")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

因為這批模擬資料的截距/斜率差異本來就很小（每條光譜只乘了一個接近 1 的隨機振幅、加了一個很小的
線性傾斜），SNV／導數對分數圖的影響在這裡不會很戲劇化；但對真實量測的 NIR 資料，SNV 或導數常常能
**大幅改善分群結果**，因為它們能去除粒徑散射造成的截距/斜率差異，讓 PCA 抓到的變異更接近真正的化學差異，
而不是物理量測條件的差異。

## 9. R 對照版本

```r
X <- as.matrix(read.csv("oils_X.csv", header = FALSE))
pca <- prcomp(X, center = TRUE, scale. = FALSE)
scores <- pca$x[, 1:2]
loadings <- pca$rotation[, 1]
summary(pca)$importance[2, ]     # 各 PC 解釋變異比例
```

`scale. = FALSE` 對應到本筆記本「不做 autoscale」的決定；`center = TRUE` 對應 mean-centering。

## 10. 對應到 Orange

Orange 的等價流程：**File/SpectraView → (Preprocess Spectra，若尚未前處理) → PCA**，
把 PCA 的 **Data** 輸出（不是 Transformed Data）接到 **Scatter Plot** 依 `oil` 上色；
把 **Components** 輸出接到 **Line Plot** 或 Orange-Spectroscopy 的 Spectra 元件，畫成 loading vs 波數。

> ⚠️ **PCA 元件裡的「Normalize variables」預設是打開的**（等同對每個波長欄位各自做 autoscale），
> 這跟本筆記本「只做 mean-centering、不做 autoscale」的選擇不一樣——請把它**取消勾選**，
> 才會是跟本筆記本、跟 R 的 `scale. = FALSE` 一致的「只置中」PCA。

> ⚠️ **匯入這份 CSV 到 Orange 時要注意角色 (role)**：`sample_id`、`group`、`instrument`、`rep`
> 這幾個中繼資料欄位如果不特別設定，Orange 會把數字欄位（如 `group`、`rep`）當成**連續變數特徵**
> 一起送進 PCA，等於多了幾個跟油品完全無關的雜訊維度。請在 File 元件裡把
> `sample_id`／`group`／`instrument`／`rep` 的角色改成 **meta**，把 `oil` 設成 **target**，
> 只留下數字波數欄位當作 PCA 真正要分析的 feature。

## 11. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算/繪圖）。

### 問題 1：PC3 在看什麼？

畫出 PC1 vs PC3 的分數圖（`np_scores[:, 0]` vs `np_scores[:, 2]`），依 `oil` 上色。
跟 PC1 vs PC2 比起來，PC3 有沒有讓某些原本重疊的油分得更開？

In [ ]:
# TODO: 請在這裡作答

### 問題 2：autoscale 對分數圖的實際影響

對 `Xc_autoscale`（第 2 節算過）重新做一次 PCA（可以用 numpy SVD 或 sklearn），
畫出分數圖，跟只做 mean-centering 的版本比較：油品之間的分群是變好、變壞，還是差不多？

In [ ]:
# TODO: 請在這裡作答

### 問題 3：如果拿掉離群樣品

第 7 節已經先回頭檢查過 `PEA_r3` 的原始光譜、確認它是量測異常並記錄了理由。
現在把它從資料中移除，重新做一次 PCA，比較解釋變異、PC2 loading，以及其他花生油重複的分數位置，
跟原本(含離群值)的結果有什麼不同？

In [ ]:
# TODO: 請在這裡作答

### 問題 4：Hotelling T² 界線敏感度

把第 7 節 Hotelling T² 計算所用的主成分數 k 改成 3，重新計算 T² 與 95% 界線，
比較哪些樣品被標記為離群樣品跟 k=2 時是否一樣。

In [ ]:
# TODO: 請在這裡作答

### 問題 5：PC1+PC2 解釋變異的計算闖關

已知 5 個主成分的特徵值分別為 `[8.2, 3.1, 0.9, 0.5, 0.3]`。
求 PC1 與 PC2 合計解釋了全部變異的百分之多少？（公式：$(\lambda_1+\lambda_2)/\sum\lambda_i \times 100\%$）

In [ ]:
# TODO: 請在這裡作答

## 12. 匯出結果 (Export)

把 PCA 分數與 T² 存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；
在本機執行則只會存檔，不會嘗試下載。

In [ ]:
export_df = meta[["sample_id", "oil", "rep"]].copy()
export_df["PC1"] = PC1
export_df["PC2"] = PC2
export_df["T2_k2"] = T2
export_df["T2_limit_k2"] = T2_limit

output_path = "nb10_pca_scores.csv"
export_df.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")
export_df

In [ ]:
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")